# Laboratory Work 4

### Load BOWS2 Images
Prepare an image array for tasks from `lab-4.pdf`.

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

In [ ]:
cover_dir = Path('BOWS2') / 'cover'

if not cover_dir.exists():
    raise FileNotFoundError(f'Directory not found: {cover_dir.resolve()}')

image_paths = sorted(cover_dir.glob('*.pgm'), key=lambda p: int(p.stem))
if not image_paths:
    raise RuntimeError(f'No .pgm files found in {cover_dir}')

images = [np.array(Image.open(path), dtype=np.uint8) for path in image_paths]
images = np.stack(images, axis=0)

print(f'Loaded images: {images.shape[0]}')
print(f'Image size: {images.shape[1:]}')
print(f'dtype: {images.dtype}')

### Step 1. Feature Extraction for Steganalysis (Method-Aware)
The notebook now supports all three methods with the same workflow: **feature extraction -> supervised classification**.
- `pair_values`: PoV statistics for bit-plane `p` (used for variant 11).
- `transition_frequency`: transition-pattern features from a bit-plane sequence.
- `run_lengths`: run-length histogram features from a bit-plane sequence.
For `transition_frequency` and `run_lengths`, serpentine scan is used by default.
For `run_lengths` with `embedding_mode='pm1'`, features are computed jointly from two bit-planes.

In [ ]:
def serpentine_flatten(arr2d: np.ndarray) -> np.ndarray:
    """Flatten a 2D array with serpentine scan (left->right, right->left, ...)."""
    h, w = arr2d.shape
    out = np.empty(h * w, dtype=arr2d.dtype)
    k = 0
    for r in range(h):
        row = arr2d[r] if (r % 2 == 0) else arr2d[r][::-1]
        out[k:k + w] = row
        k += w
    return out


def bit_plane_sequence(image: np.ndarray, p: int, use_serpentine: bool = True) -> np.ndarray:
    """Return a 1D binary sequence of bit-plane p from a uint8 grayscale image."""
    if not (1 <= p <= 8):
        raise ValueError('p must be in [1, 8] for uint8 images')
    bits2d = ((image.astype(np.uint8, copy=False) >> (p - 1)) & 1).astype(np.uint8)
    return serpentine_flatten(bits2d) if use_serpentine else bits2d.reshape(-1)


def transition_frequency_feature_vector(
    image: np.ndarray,
    p: int = 1,
    use_serpentine: bool = True,
) -> np.ndarray:
    """Build transition-frequency features from a bit-plane sequence."""
    seq = bit_plane_sequence(image, p=p, use_serpentine=use_serpentine)
    if seq.size < 2:
        return np.zeros(5, dtype=np.float64)

    pairs = (seq[:-1] << 1) | seq[1:]
    counts = np.bincount(pairs, minlength=4).astype(np.float64)  # 00, 01, 10, 11
    total = counts.sum()
    probs = counts / total if total > 0 else np.zeros(4, dtype=np.float64)
    transition_rate = probs[1] + probs[2]
    return np.concatenate([[transition_rate], probs], axis=0)


def _run_length_hist(seq: np.ndarray, max_run: int = 8) -> np.ndarray:
    """Normalized histogram of run lengths (1..max_run, with max_run as a tail bin)."""
    if seq.size == 0:
        return np.zeros(max_run, dtype=np.float64)

    change_idx = np.flatnonzero(np.diff(seq)) + 1
    run_starts = np.concatenate(([0], change_idx))
    run_ends = np.concatenate((change_idx, [seq.size]))
    run_lengths = run_ends - run_starts

    bins = np.minimum(run_lengths, max_run)
    hist = np.bincount(bins, minlength=max_run + 1).astype(np.float64)[1:max_run + 1]
    total_runs = hist.sum()
    return hist / total_runs if total_runs > 0 else hist


def run_length_feature_vector(
    image: np.ndarray,
    p: int = 1,
    use_serpentine: bool = True,
    max_run: int = 8,
    embedding_mode: str = 'lsb',
) -> np.ndarray:
    """Build run-length features; for pm1 combine two neighboring bit-planes."""
    if embedding_mode not in {'lsb', 'pm1'}:
        raise ValueError("embedding_mode must be 'lsb' or 'pm1'")

    planes = [p]
    if embedding_mode == 'pm1':
        p2 = p + 1 if p < 8 else p - 1
        planes = [p, p2]

    feats = []
    for pp in planes:
        seq = bit_plane_sequence(image, p=pp, use_serpentine=use_serpentine)
        hist_all = _run_length_hist(seq, max_run=max_run)
        hist_zero = _run_length_hist(seq[seq == 0], max_run=max_run)
        hist_one = _run_length_hist(seq[seq == 1], max_run=max_run)
        feats.append(np.concatenate([hist_all, hist_zero, hist_one], axis=0))

    return np.concatenate(feats, axis=0)


def pair_value_feature_vector(image: np.ndarray, p: int = 2) -> np.ndarray:
    """Build a PoV feature vector for one grayscale uint8 image and bit-plane p."""
    if not (1 <= p <= 8):
        raise ValueError('p must be in [1, 8] for uint8 images')

    image_u8 = image.astype(np.uint8, copy=False)
    hist = np.bincount(image_u8.ravel(), minlength=256).astype(np.float64)

    mask = 1 << (p - 1)
    left = np.arange(256, dtype=np.uint16)
    right = left ^ mask

    pair_selector = left < right
    left = left[pair_selector]
    right = right[pair_selector]

    left_counts = hist[left]
    right_counts = hist[right]
    pair_counts = left_counts + right_counts

    rel_pair_diff = np.divide(
        left_counts - right_counts,
        pair_counts,
        out=np.zeros_like(pair_counts),
        where=pair_counts > 0,
    )
    pair_weight = pair_counts / pair_counts.sum()
    return np.concatenate([rel_pair_diff, pair_weight], axis=0)


def extract_features(
    images: np.ndarray,
    method: str = 'pair_values',
    p: int = 2,
    use_serpentine: bool = True,
    max_run: int = 8,
    embedding_mode: str = 'lsb',
) -> np.ndarray:
    """Build a feature matrix for one of three steganalysis methods."""
    if method == 'pair_values':
        return np.vstack([pair_value_feature_vector(img, p=p) for img in images])
    if method == 'transition_frequency':
        return np.vstack([
            transition_frequency_feature_vector(img, p=p, use_serpentine=use_serpentine)
            for img in images
        ])
    if method == 'run_lengths':
        return np.vstack([
            run_length_feature_vector(
                img,
                p=p,
                use_serpentine=use_serpentine,
                max_run=max_run,
                embedding_mode=embedding_mode,
            )
            for img in images
        ])
    raise ValueError("method must be one of: 'pair_values', 'transition_frequency', 'run_lengths'")


def extract_pair_features(images: np.ndarray, p: int = 2) -> np.ndarray:
    """Backward-compatible alias for PoV features."""
    return extract_features(images, method='pair_values', p=p)

In [ ]:
BIT_PLANE = 2
STEGANALYSIS_METHOD = 'pair_values'
EMBEDDING_MODE = 'lsb'  # use 'pm1' for +-1 embedding scenarios
USE_SERPENTINE = True
MAX_RUN = 8

X_features = extract_features(
    images,
    method=STEGANALYSIS_METHOD,
    p=BIT_PLANE,
    use_serpentine=USE_SERPENTINE,
    max_run=MAX_RUN,
    embedding_mode=EMBEDDING_MODE,
)

print(f'Method: {STEGANALYSIS_METHOD}')
print(f'Bit-plane p: {BIT_PLANE}')
print(f'Feature matrix shape: {X_features.shape}')
print(f'Features per image: {X_features.shape[1]}')
print('First 10 features of the first image:')
print(np.round(X_features[0, :10], 6))

### Step 2. Steganographic System Simulation
Simulate LSB replacement for the first `K/2` images by filling a fraction `q` of bit-plane `p` with independent uniform white noise bitstreams. For variant 11 (odd), embedding positions are selected sequentially.

In [ ]:
def simulate_lsb_replacement(
    images: np.ndarray,
    p: int,
    q: float,
    K: int | None = None,
    position_mode: str = "sequential",
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Return (images_sim, y) where:
    - images_sim: first K/2 images are modified, second K/2 are unchanged
    - y: labels for the first K images (1 = embedded, 0 = clean)
    """
    if not (1 <= p <= 8):
        raise ValueError("p must be in [1, 8] for uint8 images")
    if not (0.0 <= q <= 1.0):
        raise ValueError("q must be in [0, 1]")
    if position_mode not in {"sequential", "random"}:
        raise ValueError("position_mode must be \"sequential\" or \"random\"")

    n_images, h, w = images.shape
    if K is None:
        K = n_images
    K = int(K)
    if K <= 0 or K > n_images:
        raise ValueError("K must be in [1, number_of_images]")

    half = K // 2
    n_pixels = h * w
    m = int(np.floor(q * n_pixels))

    images_sim = images[:K].copy()
    y = np.zeros(K, dtype=np.uint8)
    y[:half] = 1

    if m == 0 or half == 0:
        return images_sim, y

    mask = np.uint8(1 << (p - 1))
    inv_mask = np.uint8(255 ^ mask)
    rng = np.random.default_rng(seed)

    if position_mode == "sequential":
        idx = np.arange(m, dtype=np.int64)
        noise = rng.integers(0, 2, size=(half, m), dtype=np.uint8)
        for i in range(half):
            flat = images_sim[i].reshape(-1)
            flat[idx] = (flat[idx] & inv_mask) | (noise[i] * mask)
    else:
        for i in range(half):
            idx = rng.choice(n_pixels, size=m, replace=False)
            noise = rng.integers(0, 2, size=m, dtype=np.uint8)
            flat = images_sim[i].reshape(-1)
            flat[idx] = (flat[idx] & inv_mask) | (noise * mask)

    return images_sim, y

In [ ]:
BIT_PLANE = 2
POSITION_MODE = "sequential"

# You can change q and K for experiments
Q_FILL = 0.3
K_IMAGES = images.shape[0]

images_sim, y = simulate_lsb_replacement(
    images=images,
    p=BIT_PLANE,
    q=Q_FILL,
    K=K_IMAGES,
    position_mode=POSITION_MODE,
    seed=42,
)

print(f"p = {BIT_PLANE}, q = {Q_FILL}, K = {K_IMAGES}")
print(f"Position mode: {POSITION_MODE}")
print(f"Simulated set shape: {images_sim.shape}")
print(f"Embedded images: {int(y.sum())}, clean images: {int((y == 0).sum())}")
print(f"Pixels modified per embedded image: {int(np.floor(Q_FILL * images.shape[1] * images.shape[2]))}")

### Step 3 Prep. Build Features and Create Train/Test Split
Compute features for the selected steganalysis method and split data using the first 70% of each class (embedded and clean) for training, with the remaining 30% for testing.

In [ ]:
if 'images_sim' not in globals() or 'y' not in globals():
    raise NameError('Run Step 2 simulation cell first to create `images_sim` and `y`.')

if 'BIT_PLANE' not in globals():
    BIT_PLANE = 2
if 'STEGANALYSIS_METHOD' not in globals():
    STEGANALYSIS_METHOD = 'pair_values'
if 'EMBEDDING_MODE' not in globals():
    EMBEDDING_MODE = 'lsb'
if 'USE_SERPENTINE' not in globals():
    USE_SERPENTINE = True
if 'MAX_RUN' not in globals():
    MAX_RUN = 8

X_features_sim = extract_features(
    images_sim,
    method=STEGANALYSIS_METHOD,
    p=BIT_PLANE,
    use_serpentine=USE_SERPENTINE,
    max_run=MAX_RUN,
    embedding_mode=EMBEDDING_MODE,
)

embedded_idx = np.where(y == 1)[0]
clean_idx = np.where(y == 0)[0]

n_train_embedded = int(np.floor(0.7 * len(embedded_idx)))
n_train_clean = int(np.floor(0.7 * len(clean_idx)))

train_idx = np.concatenate([
    embedded_idx[:n_train_embedded],
    clean_idx[:n_train_clean],
])
test_idx = np.concatenate([
    embedded_idx[n_train_embedded:],
    clean_idx[n_train_clean:],
])

X_train = X_features_sim[train_idx]
y_train = y[train_idx]
X_test = X_features_sim[test_idx]
y_test = y[test_idx]

print(f'Method: {STEGANALYSIS_METHOD}')
print(f'Bit-plane p: {BIT_PLANE}')
print(f'Feature matrix shape: {X_features_sim.shape}')
print(f'Embedded samples: {len(embedded_idx)}, clean samples: {len(clean_idx)}')
print(f'Train size: {len(train_idx)} (embedded={n_train_embedded}, clean={n_train_clean})')
print(f'Test size: {len(test_idx)}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

### Step 3. Train Classifier and Evaluate Quality
Train a binary classifier on the 70/30 split and report both metrics: Accuracy and F1. Also build the confusion matrix.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

if not all(name in globals() for name in ['X_train', 'y_train', 'X_test', 'y_test']):
    raise NameError('Run Step 3 prep cell first to create X_train, y_train, X_test, y_test.')

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42)),
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy: {acc:.6f}')
print(f'F1-score: {f1:.6f}')
print('Confusion matrix (rows=true, cols=pred):')
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['clean (0)', 'embedded (1)'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.show()

### Steps 4-5. Evaluate on Test Set for q = 0.1..1.0
Repeat steps 2-4 for `q` from 0.1 to 1.0 with step 0.1. Store both metrics (`Accuracy`, `F1-score`) and plot metric curves versus `q`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

required = ['images', 'simulate_lsb_replacement', 'extract_features']
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f'Missing required objects: {missing}. Run previous cells first.')

if 'BIT_PLANE' not in globals():
    BIT_PLANE = 2
if 'STEGANALYSIS_METHOD' not in globals():
    STEGANALYSIS_METHOD = 'pair_values'
if 'EMBEDDING_MODE' not in globals():
    EMBEDDING_MODE = 'lsb'
if 'USE_SERPENTINE' not in globals():
    USE_SERPENTINE = True
if 'MAX_RUN' not in globals():
    MAX_RUN = 8

POSITION_MODE = 'sequential'  # variant 11 is odd
K_IMAGES = images.shape[0]
Q_VALUES = np.round(np.arange(0.1, 1.01, 0.1), 1)

results = []

for q in Q_VALUES:
    images_sim_q, y_q = simulate_lsb_replacement(
        images=images,
        p=BIT_PLANE,
        q=float(q),
        K=K_IMAGES,
        position_mode=POSITION_MODE,
        seed=42,
    )

    X_q = extract_features(
        images_sim_q,
        method=STEGANALYSIS_METHOD,
        p=BIT_PLANE,
        use_serpentine=USE_SERPENTINE,
        max_run=MAX_RUN,
        embedding_mode=EMBEDDING_MODE,
    )

    embedded_idx = np.where(y_q == 1)[0]
    clean_idx = np.where(y_q == 0)[0]

    n_train_embedded = int(np.floor(0.7 * len(embedded_idx)))
    n_train_clean = int(np.floor(0.7 * len(clean_idx)))

    train_idx = np.concatenate([
        embedded_idx[:n_train_embedded],
        clean_idx[:n_train_clean],
    ])
    test_idx = np.concatenate([
        embedded_idx[n_train_embedded:],
        clean_idx[n_train_clean:],
    ])

    X_train_q = X_q[train_idx]
    y_train_q = y_q[train_idx]
    X_test_q = X_q[test_idx]
    y_test_q = y_q[test_idx]

    clf_q = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=42)),
    ])
    clf_q.fit(X_train_q, y_train_q)
    y_pred_q = clf_q.predict(X_test_q)

    acc_q = accuracy_score(y_test_q, y_pred_q)
    f1_q = f1_score(y_test_q, y_pred_q)

    results.append({
        'q': float(q),
        'accuracy': float(acc_q),
        'f1_score': float(f1_q),
    })

results_df = pd.DataFrame(results).sort_values('q').reset_index(drop=True)
print(f'Method: {STEGANALYSIS_METHOD}, p={BIT_PLANE}, embedding_mode={EMBEDDING_MODE}')
print(results_df.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.plot(results_df['q'], results_df['accuracy'], marker='o', label='Accuracy')
plt.plot(results_df['q'], results_df['f1_score'], marker='s', label='F1-score')
plt.xticks(Q_VALUES)
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.xlabel('q (embedding fraction)')
plt.ylabel('Metric value')
plt.title('Steganalysis Quality vs Embedding Fraction q')
plt.legend()
plt.show()